# Capítulo 10 · Evaluación y límites

Cuatro preguntas al GPT del capítulo 7: ¿memoriza o generaliza?, ¿funciona fuera de su terreno?, ¿dice la verdad o solo escribe bien?, ¿qué más ha aprendido de sus datos? PyTorch en CPU. El cuaderno completo tarda unos cuatro minutos.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Constantes y modelo

En Colab, la celda descarga el modelo, la novela y el texto de los capítulos 1 a 4 si no están.

In [1]:
"""Capítulo 10: qué demuestra una cifra. PyTorch en CPU."""
import math
import re
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

RAIZ = Path(".")
MODELO = RAIZ / "material" / "gpt_quijote.pt"
TEXTO = RAIZ / "material" / "quijote.txt"
OTRO_TEXTO = RAIZ / "material" / "corpus_capitulos_1_4.txt"
for archivo in (MODELO, TEXTO, OTRO_TEXTO):  # en Colab, los descargamos del repositorio
    if not archivo.exists():
        import urllib.request
        archivo.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/" + archivo.name, archivo)
SEMILLA = 1337
PEQUENO = dict(contexto=32, dimension=32, cabezas=2, capas=2)
CARACTERES_PEQUENO = 2000  # texto con el que se entrena el modelo pequeño
SECUENCIAS = 64            # textos de 2000 caracteres generados para los sesgos
HECHOS = [("Dulcinea del ", "Toboso"), ("Sancho ", "Panza"),
          ("don Quijote de la ", "Mancha"), ("el cura y el ", "barbero"),
          ("el Caballero de la Triste ", "Figura"),
          ("el bachiller Sansón ", "Carrasco"), ("los molinos de ", "viento"),
          ("Cide Hamete ", "Benengeli")]
PAREJAS = [("caballero", "dama"), ("padre", "madre"), ("rey", "reina"),
           ("señor", "señora"), ("él", "ella"), ("hombre", "mujer")]


# --- El modelo de los capítulos 6 a 9 ---------------------------------------

class Cabeza(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["contexto"]
        tamano = C // config["cabezas"]
        self.consulta = nn.Linear(C, tamano, bias=False)
        self.clave = nn.Linear(C, tamano, bias=False)
        self.valor = nn.Linear(C, tamano, bias=False)
        self.register_buffer("mascara", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.consulta(x), self.clave(x), self.valor(x)
        puntuaciones = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        futuro = self.mascara[:T, :T] == 0
        puntuaciones = puntuaciones.masked_fill(futuro, float("-inf"))
        return F.softmax(puntuaciones, dim=-1) @ v


class Bloque(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.cabezas = nn.ModuleList(
            [Cabeza(config) for _ in range(config["cabezas"])])
        self.proyeccion = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norma1, self.norma2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norma1(x)
        atencion = torch.cat([cabeza(h) for cabeza in self.cabezas], dim=-1)
        x = x + self.proyeccion(atencion)
        return x + self.mlp(self.norma2(x))


class GPT(nn.Module):
    def __init__(self, vocabulario, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.embedding_simbolos = nn.Embedding(vocabulario, C)
        self.embedding_posiciones = nn.Embedding(config["contexto"], C)
        self.bloques = nn.Sequential(
            *[Bloque(config) for _ in range(config["capas"])])
        self.norma = nn.LayerNorm(C)
        self.salida = nn.Linear(C, vocabulario)

    def forward(self, ids):
        posiciones = torch.arange(ids.shape[1])
        x = self.embedding_simbolos(ids) + self.embedding_posiciones(posiciones)
        return self.salida(self.norma(self.bloques(x)))


def cargar_modelo(ruta=MODELO):
    guardado = torch.load(ruta)
    simbolos = guardado["simbolos"]
    modelo = GPT(len(simbolos), guardado["config"])
    modelo.load_state_dict(guardado["modelo"])
    return modelo, simbolos, {c: i for i, c in enumerate(simbolos)}

## 2. Herramientas de medida

In [2]:
# --- Herramientas de medida -------------------------------------------------

def perdida_media(modelo, x, y):
    logits = modelo(x)
    return F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))


@torch.no_grad()
def perdida_texto(modelo, datos, grupo=256):
    """Pérdida media en todo un texto, trozo a trozo (capítulo 7)."""
    T = modelo.config["contexto"]
    n = (len(datos) - 1) // T
    x, y = datos[:n * T].view(n, T), datos[1:n * T + 1].view(n, T)
    total = 0.0
    for i in range(0, n, grupo):
        total += perdida_media(modelo, x[i:i + grupo],
                               y[i:i + grupo]).item() * len(x[i:i + grupo])
    return total / n


@torch.no_grad()
def continuar(modelo, textos, n, simbolos, a_id, temperatura=0,
              generador=None):
    """Continúa a la vez varios textos de la misma longitud."""
    contexto = modelo.config["contexto"]
    ids = torch.tensor([[a_id[c] for c in texto] for texto in textos])
    for _ in range(n):
        logits = modelo(ids[:, -contexto:])[:, -1]
        if temperatura == 0:
            siguiente = logits.argmax(dim=-1, keepdim=True)
        else:
            probabilidades = F.softmax(logits / temperatura, dim=-1)
            siguiente = torch.multinomial(probabilidades, 1,
                                          generator=generador)
        ids = torch.cat([ids, siguiente], dim=1)
    return ["".join(simbolos[i] for i in fila[-n:]) for fila in ids.tolist()]

## 3. ¿Memoriza o generaliza?

In [3]:
# --- 1. ¿Memoriza o generaliza? ---------------------------------------------

def repetir_trozos(modelo, texto, simbolos, a_id, generador, muestras=500,
                   pista=32):
    """¿Sigue el modelo un trozo del texto tal como sigue en el original?"""
    inicios = torch.randint(len(texto) - 2 * pista, (muestras,),
                            generator=generador).tolist()
    pistas = [texto[i:i + pista] for i in inicios]
    originales = [texto[i + pista:i + 2 * pista] for i in inicios]
    generados = continuar(modelo, pistas, pista, simbolos, a_id)
    exactos = sum(g == o for g, o in zip(generados, originales))
    iguales = sum(a == b for g, o in zip(generados, originales)
                  for a, b in zip(g, o))
    return exactos / muestras, iguales / (muestras * pista)


def entrenar_pequeno(datos, validacion, vocabulario, pasos=2000,
                     intervalo=250):
    """Un modelo pequeño que se entrena con muy poco texto."""
    torch.manual_seed(SEMILLA)
    modelo = GPT(vocabulario, PEQUENO)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=3e-3,
                                    weight_decay=0.0)
    generador = torch.Generator().manual_seed(SEMILLA)
    T, historial = PEQUENO["contexto"], []
    for paso in range(pasos + 1):
        if paso % intervalo == 0:
            historial.append((paso, perdida_texto(modelo, datos),
                              perdida_texto(modelo, validacion)))
        if paso == pasos:
            break
        inicios = torch.randint(len(datos) - T - 1, (32,), generator=generador)
        x = torch.stack([datos[i:i + T] for i in inicios])
        y = torch.stack([datos[i + 1:i + T + 1] for i in inicios])
        perdida = perdida_media(modelo, x, y)
        optimizador.zero_grad(set_to_none=True)
        perdida.backward()
        optimizador.step()
    return modelo, historial

## 4. Fuera de su terreno, hechos y sesgos

In [4]:
# --- 2. Fuera de su terreno -------------------------------------------------

def limpiar(texto, a_id):
    """Quita los caracteres que el modelo no conoce y cuenta cuántos son."""
    desconocidos = [c for c in texto if c not in a_id]
    limpio = "".join(c for c in texto if c in a_id)
    return limpio, len(desconocidos), sorted(set(desconocidos))


# --- 3. Hechos y 4. sesgos --------------------------------------------------

@torch.no_grad()
def probabilidad(modelo, inicio, continuacion, a_id):
    """Probabilidad que da el modelo a una continuación completa."""
    ids = torch.tensor([[a_id[c] for c in inicio + continuacion]])
    logits = modelo(ids)[0, len(inicio) - 1:-1]
    objetivos = ids[0, len(inicio):]
    logaritmos = F.log_softmax(logits, dim=-1)[range(len(objetivos)),
                                                objetivos]
    return logaritmos.sum().exp().item()


def comprobar_hechos(modelo, entrenamiento, simbolos, a_id):
    """¿Completa el modelo cada expresión como en la novela?"""
    filas = []
    for pista, respuesta in HECHOS:
        con_pista = entrenamiento.count(pista)
        con_respuesta = entrenamiento.count(pista + respuesta)
        salida = continuar(modelo, [pista], len(respuesta), simbolos, a_id)[0]
        filas.append((pista + respuesta, con_respuesta, con_pista, salida,
                      probabilidad(modelo, pista, respuesta, a_id),
                      probabilidad(modelo, pista, salida, a_id)))
    return filas


def contar(texto, palabra):
    return len(re.findall(r"\b" + palabra + r"\b", texto.lower()))


def comparar_parejas(novela, generado):
    """Qué parte de cada pareja se lleva su primera palabra."""
    filas = []
    for a, b in PAREJAS:
        en_novela = (contar(novela, a), contar(novela, b))
        en_generado = (contar(generado, a), contar(generado, b))
        filas.append(((a, b), en_novela, en_generado))
    return filas


def proporcion(par):
    return par[0] / (par[0] + par[1])

## 5. El experimento completo

Tarda unos cuatro minutos.

In [5]:
# --- El experimento completo ------------------------------------------------

def ejecutar():
    torch.set_num_threads(4)
    generador = torch.Generator().manual_seed(SEMILLA)
    modelo, simbolos, a_id = cargar_modelo()
    texto = TEXTO.read_text(encoding="utf-8")
    a, b = int(0.8 * len(texto)), int(0.9 * len(texto))
    entrenamiento, validacion, prueba = texto[:a], texto[a:b], texto[b:]
    datos = {nombre: torch.tensor([a_id[c] for c in bloque]) for nombre, bloque
             in (("entrenamiento", entrenamiento), ("validación", validacion),
                 ("prueba", prueba))}
    resultados = {}

    print("1. ¿Memoriza o generaliza?")
    resultados["pérdidas"] = {nombre: perdida_texto(modelo, d)
                              for nombre, d in datos.items()}
    print("   Pérdidas:", resultados["pérdidas"])
    for nombre, bloque in (("entrenamiento", entrenamiento),
                           ("prueba", prueba)):
        resultados["repetir " + nombre] = repetir_trozos(
            modelo, bloque, simbolos, a_id, generador)
        print("   Repite trozos de", nombre, resultados["repetir " + nombre])
    inicio = "En un lugar de la Mancha, de cuyo nombre no"
    print("  ", repr(inicio + continuar(modelo, [inicio], 40, simbolos,
                                        a_id)[0]))
    desde, hasta = 200000, 200000 + CARACTERES_PEQUENO
    pequeno, historial = entrenar_pequeno(datos["entrenamiento"][desde:hasta],
                                          datos["validación"][:50000],
                                          len(simbolos))
    resultados["pequeño"] = historial
    print("   Modelo pequeño con", CARACTERES_PEQUENO, "caracteres:",
          historial[-1])
    for i in (0, 900, 1800):
        pista = entrenamiento[desde + i:desde + i + 32]
        original = entrenamiento[desde + i + 32:desde + i + 72]
        salida = continuar(pequeno, [pista], 40, simbolos, a_id)[0]
        print("   ", repr(pista), "->", repr(salida), "· original:",
              repr(original))
    pista = prueba[1000:1032]
    print("    Texto nuevo:", repr(pista), "->", repr(continuar(
        pequeno, [pista], 40, simbolos, a_id)[0]))

    print("2. Fuera de su terreno")
    otro, n_desconocidos, desconocidos = limpiar(
        OTRO_TEXTO.read_text(encoding="utf-8"), a_id)
    resultados["fuera"] = perdida_texto(
        modelo, torch.tensor([a_id[c] for c in otro]))
    print("   Pérdida en los capítulos 1 a 4 de este libro: %.3f" %
          resultados["fuera"], "·", n_desconocidos, "caracteres desconocidos:",
          "".join(desconocidos))

    print("3. Hechos")
    resultados["hechos"] = comprobar_hechos(modelo, entrenamiento, simbolos,
                                            a_id)
    for fila in resultados["hechos"]:
        frase, veces, con_pista, salida, p_bien, p_salida = fila
        print("   %-33s %4d de %4d -> %-9s %6.2f %% · lo escrito %6.2f %%" % (
            frase, veces, con_pista, salida, 100 * p_bien, 100 * p_salida))

    print("4. Sesgos")
    comienzos = ["En un lugar de la Mancha"] * SECUENCIAS
    generado = "\n".join(continuar(modelo, comienzos, 2000, simbolos, a_id,
                                   temperatura=1.0, generador=generador))
    resultados["parejas"] = comparar_parejas(entrenamiento, generado)
    for par, en_novela, en_generado in resultados["parejas"]:
        print("   %-16s novela %-11s %3.0f %% · generado %-9s %3.0f %%" % (
            "/".join(par), en_novela, 100 * proporcion(en_novela),
            en_generado, 100 * proporcion(en_generado)))
    return resultados

In [6]:
resultados = ejecutar()

1. ¿Memoriza o generaliza?
   Pérdidas: {'entrenamiento': 1.4318924301161777, 'validación': 1.4775648554563885, 'prueba': 1.4737562778902156}
   Repite trozos de entrenamiento (0.0, 0.1245625)
   Repite trozos de prueba (0.0, 0.123)
   'En un lugar de la Mancha, de cuyo nombre no se había de ser de la\nmano de la caball'
   Modelo pequeño con 2000 caracteres: (2000, 0.2394842654466629, 7.014200296047861)
    'n en las entrañas\ndel eterno olv' -> 'ido.\n\nY, viéni solvidos cuerpo a la tier' · original: 'ido.\n\nY, volviéndose a don Quijote y a l'
    ', cual lo pudieran\nmostrar bien ' -> 'esos papeles que estáis mirando, si él n' · original: 'esos papeles que estáis mirando, si él n'
    ' en semejantes despeñaderos; que' -> ' ya sé yo, y los que aquí venimos,\nla hi' · original: ' ya sé yo, y los que aquí venimos,\nla hi'
    Texto nuevo: 'rzas de\nmis desgracias. Yo, Sanc' -> 'uar, de impos que estáis mirando, si él '
2. Fuera de su terreno
   Pérdida en los capítulos 1 a 4 de este 

## Ejercicios

1. Explica con un ejemplo propio la diferencia entre memorizar y generalizar. ¿Por qué hace falta un conjunto de prueba para distinguirlas?

2. Las pérdidas de nuestro modelo son 1,432 en entrenamiento y 1,474 en prueba. ¿Qué concluyes? ¿Qué diferencia te haría sospechar que memoriza?

3. El modelo pequeño termina con una pérdida de 7,014 en texto nuevo, peor que la predicción uniforme. ¿Cómo es posible que un modelo entrenado sea peor que no saber nada?

4. Entrena el modelo pequeño con 20 000 caracteres en lugar de 2000, cambiando la constante `CARACTERES_PEQUENO`. ¿Cómo cambian las dos pérdidas finales? ¿Sigue memorizando?

5. De los caracteres de los capítulos 1 a 4 de este libro, 422 son desconocidos para el modelo. ¿Qué porcentaje del texto es? ¿Por qué no podemos simplemente ignorarlos al hablar de lo bien que funciona el modelo con ese texto?

6. Según la tabla del capítulo, después de Sancho la novela escribe Panza 197 veces de cada 692. ¿Qué porcentaje es? ¿Por qué no basta con que una expresión aparezca muchas veces para que el modelo la aprenda?

7. Añade a `HECHOS` tres expresiones más de la novela, comprueba cuántas veces aparecen y qué escribe el modelo. ¿Se cumple lo que hemos visto? (Cada comienzo debe terminar en espacio y usar solo caracteres que el modelo conoce; para probarlo no hace falta ejecutar todo el programa, basta con llamar a `comprobar_hechos`.)

8. Si el modelo respetara la proporción de la novela, con un 6 % de damas, ¿qué probabilidad habría de no ver ninguna dama entre 33 casos? (Pista: 0,94^33.) ¿Demuestra el texto generado que el modelo exagera ese desequilibrio? Cambia después `SECUENCIAS` a 640, diez veces más texto, y vuelve a ejecutar el programa (tarda unos veinticinco minutos). ¿Qué cambia en la tabla del capítulo?

9. El acierto exacto da un cero a `aeicilid` como respuesta a dulcinea al revés. Propón una medida que dé puntos parciales y calcúlala para esta respuesta. ¿Qué ventajas e inconvenientes tiene?

10. Informe de fallos. Elige un modelo de los capítulos anteriores, por ejemplo el asistente del capítulo 9, y escribe un informe de fallos con al menos tres tipos de fallo, cada uno con su criterio, su medida, su resultado y su alcance.

## Referencias

Inspirado en el vídeo [Deep Dive into LLMs like ChatGPT](https://www.youtube.com/watch?v=7xTGNNLPyMI), de Andrej Karpathy, y en [Extracting Training Data from Large Language Models](https://arxiv.org/abs/2012.07805) (Carlini y colaboradores, 2021). Texto de *Don Quijote de la Mancha*, de dominio público. Código y experimentos propios.